- Ficher source : aire_covoiturage_24-12-2024.csv
- Fichier de sorti : stg_covoit_aires.csv
- Date de création : 13/11/2025
- Dernière modification : 17/11/2025

- Version(s) : 
    - 1 - 13/11/2025 : Nettoyage du fichier et création du csv "covoit_aires"
    - 2 - 17/11/2025 : Renommage du fichier et création du csv "stg_covoit_aires"

In [2]:
#Librairie(s) utilisée(s)
import pandas as pd

In [3]:
#Création du dataframe à partir du fichier csv
df = pd.read_csv(r'C:\Users\justi\OneDrive\Je-deviens-Data-Analyst\JEDHA\00_Certif\bloc_6\01_data\01_data_bronze\aire_covoiturage_24-12-2024.csv', 
                 delimiter=';')
df.head()

,gml_id,commune,insee,nom,adresse,type,places,duree_minu,place_pmr,goudronnee,eclairage,panneau,intermodal,commentair,borne_rech,X_LB93,Y_LB93,LATITUDE,LONGITUDE
0,aire_covoiturage.40,Pipriac,35219.0,La Secouette,Echangeur D177 x D59,aire covoiturage,12.0,NaN,NaN,True,False,False,False,NaN,False,331227.5406,6.753881e+06,47.781732,-1.927601
1,aire_covoiturage.77,Étrelles,35109.0,Piquet,Piquet,aire covoiturage,80.0,NaN,True,True,True,True,False,NaN,False,387384.0135,6.783032e+06,48.072834,-1.199458
2,aire_covoiturage.51,Rieux,56194.0,Le Chêne des Bouteilles,Croisement D20xD775,aire covoiturage,15.0,NaN,NaN,True,False,True,False,NaN,False,313493.7006,6.736172e+06,47.612481,-2.148498
3,aire_covoiturage.53,Hirel,35132.0,Cimetière,D155,aire covoiturage,30.0,NaN,NaN,True,False,True,False,NaN,False,346478.2466,6.844771e+06,48.606447,-1.798030
4,aire_covoiturage.59,Saint-Père,35306.0,Le Fort Saint-Père,Echangeur de Châteauneuf d'Ille-et-Vilaine,aire covoiturage,52.0,NaN,NaN,True,False,True,False,NaN,False,336965.8896,6.840780e+06,48.565343,-1.923428


In [4]:
df.shape

(183, 19)

In [5]:
df.count()

gml_id        183
commune       183
insee         182
nom           165
adresse       171
type          166
places        162
duree_minu      1
place_pmr      75
goudronnee    155
eclairage     159
panneau       155
intermodal    133
commentair     28
borne_rech    133
X_LB93        183
Y_LB93        183
LATITUDE      183
LONGITUDE     183
dtype: int64

In [6]:
df.count().isna() #Aucune colonnes null

gml_id        False
commune       False
insee         False
nom           False
adresse       False
type          False
places        False
duree_minu    False
place_pmr     False
goudronnee    False
eclairage     False
panneau       False
intermodal    False
commentair    False
borne_rech    False
X_LB93        False
Y_LB93        False
LATITUDE      False
LONGITUDE     False
dtype: bool

In [7]:
#Création de la colonne 'MVP' pour cibler les communes cibles et filtrer le dataset dessus
MVP = [35032, 35058, 35196, 35065, 35275, 35144, 35266, 35315, 35278, 35245, 35208, 35353, 35120, 35210, 35352, 35059, 35250, 35079, 35363, 35051, 35055, 35076, 35189, 35066, 35204, 35022, 35351, 35088, 35080, 35131, 35001, 35206, 35081, 35139, 35334, 35216, 35024, 35240, 35039, 35047, 35281, 35180]

df['MVP'] = df['insee'].apply(lambda x: 'oui' if x in MVP else 'non')

df['MVP'].describe()

count     183
unique      2
top       non
freq      165
Name: MVP, dtype: object

In [8]:
#Création du mask pour filtrer sur le département de l'Ille-et-Vilaine + 'aire_covoiturage.190' qui correspond à une aire de covoiturage détruite
mask = (df['gml_id'] != 'aire_covoiturage.190') & (df['MVP'] == 'oui')
df = df[mask]

df.shape

(17, 20)

In [9]:
#Identification des colonnes à garder
keep_columns = ["gml_id","insee","nom","places","commentair","LATITUDE","LONGITUDE"]
df = df.loc[:,keep_columns]

#Renommage de la colonne INSEE
df.rename(columns={"gml_id": "covoit_id", "insee": "code_geo", "commentair" : "commentaire", "LATITUDE": "latitude", "LONGITUDE": "longitude"}, inplace=True)

df.head()

,covoit_id,code_geo,nom,places,commentaire,latitude,longitude
42,aire_covoiturage.136,35047.0,La Louvière,20.0,NaN,48.026424,-1.767333
43,aire_covoiturage.143,35088.0,La Lande du feu,25.0,NaN,47.977381,-1.560628
44,aire_covoiturage.139,35066.0,Fontenay,20.0,NaN,48.043133,-1.695399
45,aire_covoiturage.113,35059.0,Parking salle de sport,55.0,NaN,48.175061,-1.726467
52,aire_covoiturage.137,35051.0,Vaux,60.0,NaN,48.140037,-1.628017


In [11]:
#Formatage des types de colonnes
for col in df.columns:
    if col == 'covoit_id' or col == "nom" or col == "commentaire":
        df[col] = df[col].astype(str)
    elif col == 'code_geo' or col == "places":
        df[col] = df[col].astype(int)
    elif col == 'latitude' or col == "longitude":
        df[col] = df[col].astype(float)
df.dtypes

covoit_id       object
code_geo         int32
nom             object
places           int32
commentaire     object
latitude       float64
longitude      float64
dtype: object

In [ ]:
#Exporte le dataset nettoyé en csv
df.to_csv("stg_covoit_aires.csv", sep=";", index=False, encoding='UTF-8')